In [6]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, f1_score, classification_report

import pandas as pnd


<h1 style="color: red;">Section 1: Data</h1>

<h2>1) Préparation de données</h2>

In [7]:
dataset =pnd.read_csv('diabetes.csv')#import
X = np.array(dataset.drop(columns=['Outcome'])) #features
y = np.array(dataset['Outcome']) #target
#spilt data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=23)


**Importance du decoupage train/test**  
Le training set sert a apprendre les poids du modele. Le test set sert a verifier si le modele generalise bien sur des donnees non vues.


<h1 style="color: red;">Section 2: Neural network avec tensorflow</h1>

In [8]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

from tensorflow.keras.optimizers import Adam

<h2>2) Modèle de réseau de neurones</h2>

**Architecture choisie**  
- Inputs du reseau : 8 variables explicatives.  
- Couche de sortie : 1 neurone, car `Outcome` est binaire.  
- Activation de sortie : `sigmoid`, pour produire une probabilite entre `0` et `1`.


In [9]:
model_nn = Sequential()
output_layer = Dense(1, input_shape=(X_train.shape[1],), activation='sigmoid')
model_nn.add(output_layer)
opt = Adam(learning_rate=0.001)
model_nn.compile(
    optimizer=opt,
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision")]
)
history = model_nn.fit(X_train, y_train, epochs=1000, verbose=0)


In [10]:
model_nn.summary()
print("Nombre total de parametres :", model_nn.count_params())


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29 (120.00 B)

 Trainable params: 9 (36.00 B)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 20 (84.00 B)

Nombre total de parametres : 9


**Role de `fit`**  
`fit` entraine le modele sur le training set en ajustant les poids et le biais pour minimiser la loss de classification.

**Calcul du nombre de parametres**  
Avec 8 inputs et 1 neurone de sortie :
- 8 poids
- 1 biais  

Donc le modele contient `9` parametres entrainables.

**Quand faire du tuning / de la regularisation**  
Le tuning est utile quand les metriques restent faibles ou instables. La regularisation devient importante en cas de surapprentissage ou si le modele memorise trop les donnees d'entrainement.


<h2>3) Prédiction en utilisant le modèle</h2>

In [11]:
yhat_nn = model_nn.predict(X_test, verbose=0)


In [12]:
yhat_nn=yhat_nn.flatten()

In [13]:
W_nn, bias_nn = model_nn.layers[0].get_weights()

# Afficher les poids et biais
print("Poids :", W_nn.flatten())
print("Biais :", bias_nn)

Poids : [ 0.07518036  0.02536188 -0.02214933 -0.00190036 -0.00076759  0.0384067
  0.59978735  0.00274361]
Biais : [-4.2488017]


In [14]:
yhat_manual_proba = (1 / (1 + np.exp(-(X_test @ W_nn + bias_nn)))).flatten()
print("Premieres probabilites manuelles :", yhat_manual_proba[:5])
print("Predictions proches de predict :", np.allclose(yhat_nn, yhat_manual_proba, atol=1e-6))


Premieres probabilites manuelles : [0.28915272 0.85119961 0.57163874 0.34363754 0.15885719]
Predictions proches de predict : True


<h2>4) Evaluation du modèle</h2>

In [15]:
yhat_train_nn = model_nn.predict(X_train, verbose=0).flatten()
yhat_train_labels = (yhat_train_nn >= 0.5).astype(int)
yhat_test_labels = (yhat_nn >= 0.5).astype(int)

print("Train accuracy:", accuracy_score(y_train, yhat_train_labels))
print("Test accuracy :", accuracy_score(y_test, yhat_test_labels))
print("Test recall   :", recall_score(y_test, yhat_test_labels))
print("Test f1-score :", f1_score(y_test, yhat_test_labels))
print("Matrice de confusion :")
print(confusion_matrix(y_test, yhat_test_labels))
print("Rapport de classification :")
print(classification_report(y_test, yhat_test_labels))


Train accuracy: 0.754071661237785
Test accuracy : 0.7922077922077922
Test recall   : 0.49056603773584906
Test f1-score : 0.6190476190476191
Matrice de confusion :
[[96  5]
 [27 26]]
Rapport de classification :
              precision    recall  f1-score   support

           0       0.78      0.95      0.86       101
           1       0.84      0.49      0.62        53

    accuracy                           0.79       154
   macro avg       0.81      0.72      0.74       154
weighted avg       0.80      0.79      0.78       154



**Interpretation des metriques**  
- `accuracy` mesure le taux global de bonnes predictions.  
- `recall` mesure la capacite a detecter les vrais cas positifs.  
- `f1-score` equilibre precision et rappel.  
- La matrice de confusion montre les vrais/faux positifs et negatifs.

**Pourquoi evaluer sur train et test**  
La comparaison train/test permet de verifier si le modele generalise correctement et de detecter un eventuel surapprentissage.


<h1>From scratch</h1>


<h2>Modèle de régression logistic from scratch avec utilisation des matrices</h2>


**Pourquoi developper la regression logistique from scratch**  
Cette partie permet de comprendre le fonctionnement interne de la classification binaire sans dependre d'une librairie de haut niveau.

**Etapes principales**  
1. Initialiser `W` et `b`.  
2. Calculer `z = X @ W + b`.  
3. Appliquer `sigmoid(z)`.  
4. Calculer l'erreur.  
5. Calculer `dW` et `db`.  
6. Mettre a jour les parametres.  
7. Convertir les probabilites en classes avec un seuil de `0.5`.


In [16]:
learning_rate = 0.0001
epochs = 1000

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Initialisation des paramètres
W = np.zeros((X_train.shape[1], 1))
b = 0.0

# Reshape y_train pour garantir les dimensions adéquates
y_train_matrix = y_train.reshape(-1, 1)
n = len(X_train)

# Entraînement (descente de gradient vectorisée)
for epoch in range(epochs):
    y_pred = sigmoid(X_train @ W + b)  # Shape: (n, 1)
    error = y_pred - y_train_matrix  # Shape: (n, 1)

    # Calcul des gradients
    dW = (1 / n) * (X_train.T @ error)  # Shape: (n_features, 1)
    db = (1 / n) * np.sum(error)  # Scalaire

    # Mise à jour des paramètres
    W -= learning_rate * dW
    b -= learning_rate * db

print("Parametres ajustes:")
print(f"W =\n{W}")
print(f"b = {b:.4f}")


Parametres ajustes:
W =
[[ 0.01286553]
 [ 0.0142827 ]
 [-0.03123407]
 [ 0.00161999]
 [ 0.00034173]
 [-0.0040858 ]
 [ 0.00070535]
 [-0.00101959]]
b = -0.0039


In [17]:
yhat_scratch_proba = sigmoid(X_test @ W + b).flatten()
yhat_scratch_labels = (yhat_scratch_proba >= 0.5).astype(int)

print("Scratch accuracy:", accuracy_score(y_test, yhat_scratch_labels))
print("Scratch recall  :", recall_score(y_test, yhat_scratch_labels))
print("Scratch f1-score:", f1_score(y_test, yhat_scratch_labels))
print(confusion_matrix(y_test, yhat_scratch_labels))


Scratch accuracy: 0.6688311688311688
Scratch recall  : 0.22641509433962265
Scratch f1-score: 0.32
[[91 10]
 [41 12]]
